Create a flattened parquet file ommitting the messages key.

In [2]:
import json
import duckdb
import pandas as pd
from pathlib import Path
import sys
sys.path.append("/workspaces/LLM-reliance/scripts")
import extractManualLabelSample as bro


file_path = Path('../data/outputs/studychat_with_reliance_label.jsonl')
cleaned_file_path = Path("../data/outputs/clean_analytical_dataset.parquet")

flat_data = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)

        llm_label = None
        if 'llm_label' in record and isinstance(record['llm_label'], dict):
            broad_label, specific_label = bro.canonicalize_label(
                record['llm_label']['label'])
            llm_label = record['llm_label'].get('label')

        rel_label = None
        rel_reason = None

        if 'reliance_label' in record and isinstance(record['reliance_label'], dict):
            rel_label = record['reliance_label'].get('label')
            rel_reason = record['reliance_label'].get('reason')

        flat_data.append({
            'userId': record.get('userId'),
            'chatId': record.get('chatId'),
            'semester': record.get('semester'),
            'topic': record.get('topic'),
            'interactionCount': record.get('interactionCount'),
            'llm_label': llm_label,
            'broad_label': broad_label,
            'specific_label': specific_label,
            'reliance_category': rel_label,
            'reliance_reason': rel_reason
        })


df = pd.DataFrame(flat_data)


df = df.dropna(subset=['reliance_category'])
df.info()
# df.to_parquet(cleaned_file_path, engine='pyarrow')

with duckdb.connect() as connection:
    display_frame = connection.execute(
        """
        SELECT *
        FROM df
        LIMIT 3;
        """
    ).df()

display_frame

<class 'pandas.DataFrame'>
RangeIndex: 16851 entries, 0 to 16850
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   userId             16851 non-null  str  
 1   chatId             16851 non-null  str  
 2   semester           16851 non-null  str  
 3   topic              16851 non-null  str  
 4   interactionCount   16851 non-null  int64
 5   llm_label          16851 non-null  str  
 6   broad_label        16851 non-null  str  
 7   specific_label     16851 non-null  str  
 8   reliance_category  16851 non-null  str  
 9   reliance_reason    16851 non-null  str  
dtypes: int64(1), str(9)
memory usage: 6.5 MB


,userId,chatId,semester,topic,interactionCount,llm_label,broad_label,specific_label,reliance_category,reliance_reason
0,613b1510-4051-7027-1ee2-8f2c72d3c64b,7363dd3f-199d-44c4-975b-52b4117fdf21,f24,a1,0,conceptual_questions>Programming Language,conceptual_questions,Programming Language,THOUGHTLESS_USE,"The student asks a very brief, context-free ho..."
1,613b1510-4051-7027-1ee2-8f2c72d3c64b,71e5ccea-04c1-4fa1-a1dd-ed847efe470d,f24,a1,0,conceptual_questions>Computer Science,conceptual_questions,Computer Science,CAUTIOUS_USE,The student is asking for an explanation of ho...
2,613b1510-4051-7027-1ee2-8f2c72d3c64b,71e5ccea-04c1-4fa1-a1dd-ed847efe470d,f24,a1,1,writing_request>Write Code,writing_request,Write Code,THOUGHTLESS_USE,"The student gives a very terse, context-free r..."


Load and analyze student grade data

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

with duckdb.connect() as connection:
    df = connection.execute("""
                            SELECT * from read_csv($fall_sem)
                            UNION ALL
                            SELECT * from read_csv($spring_sem)
                            """,
                            {"fall_sem": str(Path("/workspaces/StudyChat/scores/f24_grades_released_normalized.csv")),
                             "spring_sem": str(Path("/workspaces/StudyChat/scores/s25_grades_released_normalized.csv"))
                             }
                            ).df()

df.info()
df

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   userId          181 non-null    str    
 1   directory_name  181 non-null    str    
 2   a1              181 non-null    float64
 3   a2              181 non-null    float64
 4   a3              181 non-null    float64
 5   a4              181 non-null    float64
 6   a5              181 non-null    float64
 7   a6              181 non-null    float64
 8   a7              181 non-null    float64
 9   e1              181 non-null    float64
 10  e2              181 non-null    float64
 11  e3              181 non-null    float64
 12  llm             181 non-null    float64
 13  no_llm          181 non-null    float64
dtypes: float64(12), str(2)
memory usage: 33.5 KB


,userId,directory_name,a1,a2,a3,a4,a5,a6,a7,e1,e2,e3,llm,no_llm
0,d1db45a0-f051-702b-f627-c33637d4528c,user_d1db45a0-f051-702b-f627-c33637d4528c,1.0,0.99,0.618182,0.967213,1.000000,1.000000,0.960396,0.9500,1.005,0.920,0.946667,0.958333
1,c18b0590-1021-70b2-a04d-7da82bf20079,user_c18b0590-1021-70b2-a04d-7da82bf20079,1.0,0.88,0.781818,0.934426,0.907692,0.876106,1.000000,0.6200,0.940,0.600,0.908571,0.720000
2,613b1510-4051-7027-1ee2-8f2c72d3c64b,user_613b1510-4051-7027-1ee2-8f2c72d3c64b,0.9,0.94,0.854545,0.934426,0.953846,1.000000,0.950495,0.8750,0.875,0.800,0.944762,0.850000
3,315b9550-4081-7033-4f93-fe2be67cdfcc,user_315b9550-4081-7033-4f93-fe2be67cdfcc,1.0,0.95,0.872727,0.934426,1.000000,0.982301,1.000000,0.7350,0.790,0.725,0.965714,0.750000
4,619bf560-4021-7015-9155-f148328587b8,user_619bf560-4021-7015-9155-f148328587b8,1.0,1.00,1.000000,0.967213,0.984615,0.982301,0.970297,0.9500,1.100,0.950,0.984762,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,411bd580-f051-7091-c64b-39f944fda7b6,user_411bd580-f051-7091-c64b-39f944fda7b6,1.0,0.98,0.800000,1.000000,1.000000,0.902655,0.992593,0.9400,0.910,0.800,0.960254,0.883333
177,61bbc5f0-2041-7061-260b-a4d10ca29e45,user_61bbc5f0-2041-7061-260b-a4d10ca29e45,1.0,1.00,0.963636,0.988889,1.000000,0.946903,1.000000,0.9300,1.050,0.910,0.985692,0.963333
178,617b2500-4071-70e9-29f1-b4c4ce63bfaa,user_617b2500-4071-70e9-29f1-b4c4ce63bfaa,1.0,1.00,0.890909,0.977778,1.000000,0.973451,0.992593,0.8650,0.870,1.000,0.980922,0.911667
179,d17b0530-40c1-7051-3339-0d2a5bb89823,user_d17b0530-40c1-7051-3339-0d2a5bb89823,1.0,0.94,0.963636,0.811111,1.000000,0.902655,0.933333,0.8700,1.010,0.920,0.928458,0.933333
